# GLM-5.1 Distillation (A100)

将 GLM-5.1 (754B MoE) 蒸馏到 Qwen3-8B

**运行前设置：** Runtime → Change runtime type → A100 GPU

---

## 0. 配置（在这里填你的 key）

In [ ]:
# ===== 必填 =====
ZHIPU_API_KEY = ""

# ===== 可选：推送到 HuggingFace Hub =====
HF_TOKEN = ""  # 填你的 HF token，留空则不推送
HF_REPO_ID = ""  # 例如 "your-username/qwen3-8b-glm5-distill"

# ===== 训练参数 =====
STUDENT_MODEL = "Qwen/Qwen3-8B"  # 或 "Qwen/Qwen3-4B"
NUM_EPOCHS = 3
BATCH_SIZE = 4  # A100 40GB 跑 8B 用 4，80GB 可以用 8
GRAD_ACCUM = 8  # 有效 batch = BATCH_SIZE * GRAD_ACCUM = 32
LEARNING_RATE = 1e-4
MAX_SEQ_LEN = 2048
LORA_R = 64
LORA_ALPHA = 128

# ===== 数据生成参数 =====
TEACHER_MODEL = "glm-5.1"
API_WORKERS = 8  # 并发数
TEMPERATURE = 0.7

## 1. 安装依赖

In [ ]:
!pip install -q zhipuai transformers datasets peft trl accelerate bitsandbytes flash-attn

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. 生成教师数据（调用 GLM-5.1 API）

如果你已有数据，跳过此步，把 JSONL 上传到 `data/train.jsonl`，格式：
```json
{"prompt": "...", "response": "..."}
```

In [ ]:
import json
import time
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from tqdm.notebook import tqdm
from zhipuai import ZhipuAI

os.makedirs("data", exist_ok=True)

# ===== 种子 prompts =====
# 这里放了 60 条覆盖多领域的种子，实际使用建议扩展到 5000+
SEED_PROMPTS = [
    # -- Code --
    {"prompt": "Explain the difference between a stack and a queue with Python examples.", "category": "code"},
    {"prompt": "Write a Python function to find the longest common subsequence of two strings.", "category": "code"},
    {"prompt": "How does garbage collection work in Python? Explain reference counting and the generational collector.", "category": "code"},
    {"prompt": "Implement a simple HTTP server in Python that handles GET and POST requests.", "category": "code"},
    {"prompt": "Write a SQL query to find the second highest salary in each department.", "category": "code"},
    {"prompt": "Implement a binary search tree with insert, delete, and search operations in Python.", "category": "code"},
    {"prompt": "Write a Python decorator that adds retry logic with exponential backoff.", "category": "code"},
    {"prompt": "Implement a simple LRU cache in Python without using functools.", "category": "code"},
    {"prompt": "Write a Python script that processes a CSV file and generates summary statistics.", "category": "code"},
    {"prompt": "Implement the observer design pattern in Python.", "category": "code"},
    {"prompt": "Write a Python function that validates an email address using regex.", "category": "code"},
    {"prompt": "Write a Python asyncio-based web scraper that respects rate limits.", "category": "code"},
    {"prompt": "Implement a thread-safe producer-consumer queue in Python.", "category": "code"},
    {"prompt": "Write a recursive descent parser for simple arithmetic expressions in Python.", "category": "code"},
    {"prompt": "Implement a trie data structure with insert, search, and prefix matching.", "category": "code"},
    {"prompt": "Write a Python context manager for database transactions with rollback on error.", "category": "code"},
    {"prompt": "Implement Dijkstra's shortest path algorithm in Python with a priority queue.", "category": "code"},
    {"prompt": "Write a Python metaclass that automatically adds logging to all methods.", "category": "code"},
    {"prompt": "Implement a simple key-value store with persistence using Python.", "category": "code"},
    {"prompt": "Write a Python generator that reads a large file in chunks without loading it all into memory.", "category": "code"},
    # -- Reasoning --
    {"prompt": "Explain the CAP theorem in distributed systems with real-world examples.", "category": "reasoning"},
    {"prompt": "What are the trade-offs between microservices and monolithic architecture?", "category": "reasoning"},
    {"prompt": "Explain how transformers work in deep learning, focusing on the attention mechanism.", "category": "reasoning"},
    {"prompt": "What is the difference between TCP and UDP? When would you use each?", "category": "reasoning"},
    {"prompt": "Explain the concept of eventual consistency and how it differs from strong consistency.", "category": "reasoning"},
    {"prompt": "What are the SOLID principles in software engineering? Give examples for each.", "category": "reasoning"},
    {"prompt": "Explain how Docker containers differ from virtual machines.", "category": "reasoning"},
    {"prompt": "What is the difference between process and thread? Explain with examples.", "category": "reasoning"},
    {"prompt": "Explain how HTTPS works, including the TLS handshake process.", "category": "reasoning"},
    {"prompt": "What is the time complexity of Dijkstra's algorithm and why?", "category": "reasoning"},
    {"prompt": "Explain the difference between supervised, unsupervised, and reinforcement learning.", "category": "reasoning"},
    {"prompt": "What is the difference between optimistic and pessimistic concurrency control?", "category": "reasoning"},
    {"prompt": "Explain how B+ trees work and why databases use them for indexing.", "category": "reasoning"},
    {"prompt": "What are the differences between REST, GraphQL, and gRPC? When to use each?", "category": "reasoning"},
    {"prompt": "Explain the concept of backpressure in streaming systems.", "category": "reasoning"},
    # -- Math --
    {"prompt": "Solve step by step: A train leaves city A at 60 km/h. Another leaves city B (300 km away) at 90 km/h toward A. When and where do they meet?", "category": "math"},
    {"prompt": "Prove that the square root of 2 is irrational.", "category": "math"},
    {"prompt": "Explain the pigeonhole principle and give 3 interesting applications.", "category": "math"},
    {"prompt": "What is the difference between P, NP, and NP-complete problems? Give examples.", "category": "math"},
    {"prompt": "Solve: How many ways can you climb n stairs if you can take 1, 2, or 3 steps at a time? Derive the recurrence.", "category": "math"},
    # -- Chinese --
    {"prompt": "请用中文解释什么是知识蒸馏，以及它在大语言模型中的应用。", "category": "chinese"},
    {"prompt": "用Python实现一个简单的聊天机器人框架，支持多轮对话。", "category": "chinese_code"},
    {"prompt": "解释MapReduce的工作原理，并给出一个词频统计的例子。", "category": "chinese"},
    {"prompt": "请解释数据库索引的原理，B+树索引和哈希索引的区别是什么？", "category": "chinese"},
    {"prompt": "用Python实现快速排序算法，并分析其时间和空间复杂度。", "category": "chinese_code"},
    {"prompt": "解释什么是微服务架构，它和单体架构相比有什么优缺点？", "category": "chinese"},
    {"prompt": "用Python实现一个支持过期时间的内存缓存系统。", "category": "chinese_code"},
    {"prompt": "解释分布式系统中的Raft共识算法的核心思想。", "category": "chinese"},
    {"prompt": "请解释Transformer模型中Multi-Head Attention的作用和实现原理。", "category": "chinese"},
    {"prompt": "用Python实现一个简单的正则表达式引擎，支持 . * + 三种操作符。", "category": "chinese_code"},
    # -- General / Creative --
    {"prompt": "Design a URL shortener system. Describe the API, storage, and how to handle high traffic.", "category": "system_design"},
    {"prompt": "Design a rate limiter. Compare token bucket, leaky bucket, and sliding window approaches.", "category": "system_design"},
    {"prompt": "How would you design a real-time chat application that scales to millions of users?", "category": "system_design"},
    {"prompt": "Explain the concept of knowledge distillation in machine learning. How is it different from model pruning and quantization?", "category": "ml"},
    {"prompt": "What are the key differences between LoRA, QLoRA, and full fine-tuning for LLMs?", "category": "ml"},
    {"prompt": "Explain how RLHF works for aligning language models. What are the alternatives like DPO?", "category": "ml"},
    {"prompt": "Write a comprehensive guide to Git branching strategies for a team of 10 developers.", "category": "general"},
    {"prompt": "Explain the differences between OAuth 2.0 and OpenID Connect with practical examples.", "category": "general"},
    {"prompt": "What are the best practices for writing production-ready Python code?", "category": "general"},
    {"prompt": "Explain event-driven architecture and compare it with request-response patterns.", "category": "general"},
]

print(f"Seed prompts: {len(SEED_PROMPTS)}")

In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful, accurate, and thoughtful assistant. "
    "Provide clear, well-structured responses. "
    "For code questions, include working examples with explanations."
)

client = ZhipuAI(api_key=ZHIPU_API_KEY)

def call_glm(prompt: str, temperature: float = TEMPERATURE) -> str | None:
    for attempt in range(3):
        try:
            resp = client.chat.completions.create(
                model=TEACHER_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt},
                ],
                temperature=temperature,
                max_tokens=2048,
            )
            return resp.choices[0].message.content
        except Exception as e:
            if attempt < 2:
                time.sleep(2 ** attempt)
            else:
                print(f"FAILED: {prompt[:60]}... | {e}")
                return None

# 断点续传
output_file = Path("data/train.jsonl")
existing = set()
if output_file.exists():
    for line in output_file.read_text().strip().split("\n"):
        if line:
            existing.add(json.loads(line).get("prompt", ""))
    print(f"已有 {len(existing)} 条数据，断点续传")

remaining = [p for p in SEED_PROMPTS if p["prompt"] not in existing]
print(f"待生成: {len(remaining)} 条")

results = []
with ThreadPoolExecutor(max_workers=API_WORKERS) as executor:
    futures = {executor.submit(call_glm, p["prompt"]): p for p in remaining}
    for future in tqdm(as_completed(futures), total=len(remaining), desc="Generating"):
        seed = futures[future]
        response = future.result()
        if response:
            results.append({
                "prompt": seed["prompt"],
                "response": response,
                "category": seed.get("category", "general"),
            })

with open(output_file, "a") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

total = len(existing) + len(results)
print(f"\n完成！共 {total} 条训练数据 -> {output_file}")

In [ ]:
# 查看几条生成的数据
with open("data/train.jsonl") as f:
    lines = f.readlines()
print(f"总数据量: {len(lines)} 条\n")
sample = json.loads(lines[0])
print(f"Prompt: {sample['prompt'][:100]}")
print(f"Response: {sample['response'][:300]}...")

## 3. LoRA SFT 训练

In [ ]:
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

# 加载数据
records = []
with open("data/train.jsonl") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        item = json.loads(line)
        records.append({
            "messages": [
                {"role": "user", "content": item["prompt"]},
                {"role": "assistant", "content": item["response"]},
            ]
        })

dataset = Dataset.from_list(records)
split = dataset.train_test_split(test_size=0.05, seed=42)
train_ds, eval_ds = split["train"], split["test"]
print(f"Train: {len(train_ds)}, Eval: {len(eval_ds)}")

In [ ]:
# 加载模型
print(f"Loading {STUDENT_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="flash_attention_2",
    trust_remote_code=True,
)

# LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# VRAM check
allocated = torch.cuda.memory_allocated() / 1e9
print(f"VRAM after loading: {allocated:.1f} GB")

In [ ]:
# 训练！
OUTPUT_DIR = "outputs/distill-lora"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    bf16=True,
    logging_steps=5,
    save_steps=100,
    save_total_limit=3,
    eval_strategy="steps",
    eval_steps=100,
    max_seq_length=MAX_SEQ_LEN,
    report_to="none",
    seed=42,
    gradient_checkpointing=True,  # 省 VRAM
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)

print("开始训练...")
trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapter 已保存到 {OUTPUT_DIR}")

## 4. 合并权重 & 导出

In [ ]:
from peft import PeftModel

# 释放训练显存
del model, trainer
torch.cuda.empty_cache()

MERGED_DIR = "models/merged"

print(f"加载 base model: {STUDENT_MODEL}")
base_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="cpu",
    trust_remote_code=True,
)

print(f"加载 LoRA adapter: {OUTPUT_DIR}")
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)

print("合并权重...")
model = model.merge_and_unload()

print(f"保存到 {MERGED_DIR}")
model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print("合并完成！")

## 5. 测试蒸馏效果

In [ ]:
del base_model
torch.cuda.empty_cache()

print(f"加载合并后的模型: {MERGED_DIR}")
model = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

test_prompts = [
    "Write a Python function to check if a string is a palindrome.",
    "用中文解释什么是梯度下降，以及学习率的作用。",
    "Design a simple task queue system with Redis. Explain the architecture.",
]

for prompt in test_prompts:
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    start = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, temperature=0.7, do_sample=True, top_p=0.9)
    elapsed = time.time() - start
    gen_tokens = out.shape[1] - inputs["input_ids"].shape[1]

    response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\n{'='*60}")
    print(f"Q: {prompt}")
    print(f"A: {response[:500]}")
    print(f"--- {gen_tokens} tokens, {elapsed:.1f}s, {gen_tokens/elapsed:.0f} tok/s ---")

## 6. 推送到 HuggingFace Hub（可选）

In [ ]:
if HF_TOKEN and HF_REPO_ID:
    from huggingface_hub import login
    login(token=HF_TOKEN)

    print(f"推送到 {HF_REPO_ID}...")
    model.push_to_hub(HF_REPO_ID)
    tokenizer.push_to_hub(HF_REPO_ID)
    print(f"完成！模型地址: https://huggingface.co/{HF_REPO_ID}")
else:
    print("跳过 Hub 推送（未设置 HF_TOKEN 和 HF_REPO_ID）")
    print("你可以手动下载 models/merged/ 目录")

## 7. 导出 GGUF 用于 Ollama/llama.cpp（可选）

In [ ]:
# 安装 llama.cpp 转换工具
!pip install -q llama-cpp-python
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp 2>/dev/null; true

# 转换为 GGUF
!python /tmp/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile models/model-f16.gguf --outtype f16

# 量化为 Q4_K_M（推理最优平衡）
!cd /tmp/llama.cpp && make -j quantize 2>/dev/null
!/tmp/llama.cpp/build/bin/llama-quantize models/model-f16.gguf models/model-Q4_K_M.gguf Q4_K_M

import os
size_gb = os.path.getsize("models/model-Q4_K_M.gguf") / 1e9
print(f"\nQ4_K_M GGUF 大小: {size_gb:.2f} GB")
print("下载此文件即可在本地用 Ollama 运行")

---
## 下载模型到本地

在 Colab 文件面板中找到 `models/model-Q4_K_M.gguf` 右键下载，或：

In [ ]:
# 打包下载
!tar czf /content/distilled-model.tar.gz models/model-Q4_K_M.gguf

from google.colab import files
files.download("/content/distilled-model.tar.gz")

print("\n本地使用方法:")
print("  1. ollama create glm5-distill -f Modelfile")
print("  2. ollama run glm5-distill")